# Deep Learning Training Monitoring: Capacity Utilization & Convergence Stability

传统的 Loss 曲线在很多情况下具有**欺骗性**，无法反映模型是否真正"学到了结构"。

为了量化：

- **容量是否被有效利用 (Capacity Utilization)**
- **收敛是否稳定 (Convergence Quality)**

我们定义了 **7 个核心结构性指标**进行分层监控。

---

# Part 1: Capacity & Structure Monitoring

> 核心问题：**我设计的参数，有多少在真正工作？**

---

## 1. Weight Participation Ratio (WPR)

**直觉：**  
衡量一层中权重更新是否"抱团"。

如果 90% 的更新集中在 10% 的参数上 → 容量利用率极低。

**定义：**

$$
\text{WPR}(\Delta W) = \frac{\left(\sum_i |\Delta w_i|\right)^2}{D \cdot \sum_i (\Delta w_i)^2}
$$

- $D$：参数总数  
- $\Delta W$：权重更新向量  

**监控意义：**

- 趋近 1 → 更新均匀，容量利用高  
- 持续 < 0.1 → 严重容量浪费  

In [1]:
import torch
import torch.nn as nn


def weight_participation_ratio(delta_w: torch.Tensor) -> float:
    """WPR ∈ (0, 1]. Higher means more uniform weight participation."""
    flat = delta_w.flatten().float()
    l1 = torch.linalg.vector_norm(flat, ord=1)
    l2_sq = torch.linalg.vector_norm(flat, ord=2) ** 2
    D = flat.numel()
    return (l1 ** 2 / (D * l2_sq)).item()


# demo
model = nn.Linear(256, 128)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

w_before = {n: p.clone() for n, p in model.named_parameters()}

x = torch.randn(32, 256)
loss = model(x).pow(2).mean()
loss.backward()
optimizer.step()

for name, param in model.named_parameters():
    delta_w = param.data - w_before[name]
    wpr = weight_participation_ratio(delta_w)
    print(f"{name:>10s}  WPR = {wpr:.4f}")

    weight  WPR = 0.6300
      bias  WPR = 0.6463


---

## 2. Feature Stable Rank (srank)

**直觉：**  
衡量特征空间的"有效维度"。

网络在低维训练，就是这个指标崩塌的表现。

**定义：**

$$
\text{srank}(H) = \frac{\|H\|_F^2}{\|H\|_2^2} = \frac{\sum_i \sigma_i^2}{\sigma_{\max}^2}
$$

- $H \in \mathbb{R}^{N \times d}$：激活矩阵  

**监控意义：**

- srank 高 → 多维表达正常  
- srank 急剧下降 → 维度坍缩  

**报警阈值：**

- srank < 0.05 × dim → collapse  

**工程注意：**

完整 SVD 复杂度 $O(N d^2)$，对 1M+ 行的激活矩阵不可行。  
实际做法：$\|H\|_F^2$ 直接累加，$\sigma_{\max}$ 用 power iteration（20 轮）估算，总复杂度 $O(N d \cdot k)$。

In [2]:
def _spectral_norm_power_iter(H: torch.Tensor, n_iters: int = 20) -> float:
    """Estimate sigma_max(H) via power iteration. O(N·d·n_iters)."""
    v = torch.randn(H.shape[1], device=H.device, dtype=H.dtype)
    v = torch.nn.functional.normalize(v, dim=0)
    for _ in range(n_iters):
        u = torch.nn.functional.normalize(H @ v, dim=0)
        v = torch.nn.functional.normalize(H.T @ u, dim=0)
    return torch.linalg.vector_norm(H @ v).item()


def feature_stable_rank(H: torch.Tensor, n_iters: int = 20) -> float:
    """Stable rank = ||H||_F^2 / sigma_max^2.
    Uses power iteration to avoid full SVD — works on 1M+ row matrices.
    """
    H = H.float()
    fro_sq = torch.linalg.matrix_norm(H, ord="fro").item() ** 2
    sigma_max = _spectral_norm_power_iter(H, n_iters)
    return fro_sq / (sigma_max ** 2 + 1e-12)


# demo
net = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 32),
)

activations = {}

def save_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

net[1].register_forward_hook(save_activation("relu"))

x = torch.randn(64, 64)
_ = net(x)

H = activations["relu"]          # shape (64, 128)
sr = feature_stable_rank(H)
dim = H.shape[1]
print(f"stable_rank = {sr:.2f}  (dim = {dim}, ratio = {sr/dim:.2%})")
if sr < 0.05 * dim:
    print("ALERT: dimensional collapse detected!")

stable_rank = 2.90  (dim = 128, ratio = 2.26%)
ALERT: dimensional collapse detected!


---

## 3. Dead Neuron Ratio (DNR)

**直觉：**  
针对 ReLU dead zone 问题。

**定义：**

$$
\text{DNR} = \frac{\text{Count}(\text{Activation} \le 0 \text{ for all samples in batch})}{\text{Total Neurons}}
$$

**监控意义：**

- DNR 高 → 大量神经元无法传播梯度  
- **按 layer 统计**：通常对每一层的激活输出分别计算 DNR；如需总体指标再做汇总（平均/最大）。
- 只看最后一层（linear 输出）只代表该层，不代表全模型。

**报警阈值：**

- DNR > 0.5 → 明显退化  
- DNR > 0.8 → 层死亡  

In [3]:
def dead_neuron_ratio(activation: torch.Tensor) -> float:
    """Fraction of neurons that output <= 0 for ALL samples in the batch.
    activation: (batch, neurons) or (batch, neurons, ...) — spatial dims are flattened.
    """
    if activation.dim() > 2:
        activation = activation.flatten(2).mean(-1)  # pool spatial dims
    dead = (activation <= 0).all(dim=0)               # True if dead across entire batch
    return dead.float().mean().item()


# demo
layer = nn.Sequential(nn.Linear(64, 256), nn.ReLU())

x = torch.randn(128, 64)
act = layer(x)  # (128, 256)

dnr = dead_neuron_ratio(act)
print(f"Dead Neuron Ratio = {dnr:.2%}")
if dnr > 0.8:
    print("ALERT: layer is effectively dead!")
elif dnr > 0.5:
    print("WARNING: significant neuron degradation.")

Dead Neuron Ratio = 0.00%


---

# Part 2: Convergence & Stability Monitoring

> 核心问题：**模型是在学知识，还是在原地做布朗运动？**

---

## 4. Gradient Noise Scale (GNS)

**直觉：**  
衡量梯度中的"信号 vs 噪声"。

**定义：**

$$
B_{\text{noise}} = \frac{\operatorname{tr}(\Sigma)}{\|G\|^2}
$$

- $\Sigma$：梯度方差  
- $G$：平均梯度  

**监控意义：**

- 大 → 噪声主导  
- 小 → 信号主导  

**工程建议：**

- GNS 大 → 增大 Batch Size 或降低 Learning Rate  
- GNS 小 → 当前训练效率高  

**高危警告：**

如果 Attention 在学噪声，该指标会出现数量级飙升。

**论文出处：**

> Sam McCandlish, Jared Kaplan, Dario Amodei, OpenAI.  
> *An Empirical Model of Large-Batch Training.* 2018.  
> https://arxiv.org/abs/1812.06162

In [4]:
def gradient_noise_scale(model: nn.Module, loss_fn, data_batches: list) -> float:
    """Estimate GNS by computing gradient variance across multiple micro-batches.
    Larger GNS → noise-dominated; smaller → signal-dominated.
    """
    grads = []
    for x, y in data_batches:
        model.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        g = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None])
        grads.append(g)

    grads = torch.stack(grads)       # (K, D)
    G_mean = grads.mean(dim=0)       # (D,)
    G_var = grads.var(dim=0)         # per-param variance

    tr_sigma = G_var.sum().item()
    g_norm_sq = (G_mean ** 2).sum().item()
    return tr_sigma / (g_norm_sq + 1e-12)


# demo
model = nn.Linear(32, 10)
loss_fn = nn.CrossEntropyLoss()

micro_batches = [(torch.randn(16, 32), torch.randint(0, 10, (16,))) for _ in range(8)]

gns = gradient_noise_scale(model, loss_fn, micro_batches)
print(f"Gradient Noise Scale = {gns:.2f}")

Gradient Noise Scale = 7.30


---

## 5. Weight-to-Update Ratio

**直觉：**  
衡量更新步长相对于权重本身的比例。

**定义：**

$$
R_l = \frac{\|\eta \cdot \Delta W_l\|_2}{\|W_l\|_2}
$$

**监控意义：**

| 区间 | 状态 |
|------|------|
| $10^{-4} \sim 10^{-2}$ | 正常 |
| $> 10^{-1}$ | 发散风险 |
| $< 10^{-5}$ | 更新停滞 |

In [5]:
def weight_to_update_ratio(w_before: torch.Tensor, w_after: torch.Tensor) -> float:
    """Ratio of update magnitude to weight magnitude.
    Healthy range: 1e-4 ~ 1e-2.
    """
    delta = (w_after - w_before).float()
    return (torch.linalg.vector_norm(delta) / torch.linalg.vector_norm(w_before.float())).item()


# demo
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

snapshots = {n: p.data.clone() for n, p in model.named_parameters()}

x = torch.randn(32, 64)
loss = nn.functional.cross_entropy(model(x), torch.randint(0, 10, (32,)))
loss.backward()
optimizer.step()

for name, param in model.named_parameters():
    ratio = weight_to_update_ratio(snapshots[name], param.data)
    status = "OK" if 1e-4 <= ratio <= 1e-2 else ("DIVERGE?" if ratio > 0.1 else "STALLED?")
    print(f"{name:>12s}  update/weight = {ratio:.2e}  [{status}]")

    0.weight  update/weight = 1.39e-02  [STALLED?]
      0.bias  update/weight = 1.40e-02  [STALLED?]
    2.weight  update/weight = 1.97e-02  [STALLED?]
      2.bias  update/weight = 2.03e-02  [STALLED?]


---

## 6. Attention Entropy

**直觉：**  
衡量 Attention 是否有效。

**定义：**

$$
H = - \sum_i p_i \log p_i
$$

**监控意义：**

- 接近 $\log N$ → 完全随机（无效 Attention）  
- 接近 0 → 过度集中  

In [8]:
import math
from torch.distributions import Categorical


def attention_entropy(attn_weights: torch.Tensor) -> torch.Tensor:
    """Per-head average entropy of attention distributions.
    attn_weights: (batch, heads, seq_q, seq_k) — already softmax'd.
    Returns: (heads,) tensor of mean entropy per head.
    """
    dist = Categorical(probs=attn_weights)      # distribution over last dim (seq_k)
    ent = dist.entropy()                         # (B, H, Q)
    return ent.mean(dim=(0, 2))                  # average over batch & query positions


# demo
B, H, Q, K = 4, 8, 16, 16
logits = torch.randn(B, H, Q, K)
attn = torch.softmax(logits, dim=-1)

ent = attention_entropy(attn)
max_ent = math.log(K)

for h in range(H):
    ratio = ent[h].item() / max_ent
    status = "uniform (ineffective)" if ratio > 0.95 else ("collapsed" if ratio < 0.1 else "healthy")
    print(f"Head {h}: entropy = {ent[h]:.3f} / {max_ent:.3f} ({ratio:.1%})  [{status}]")

Head 0: entropy = 2.340 / 2.773 (84.4%)  [healthy]
Head 1: entropy = 2.339 / 2.773 (84.4%)  [healthy]
Head 2: entropy = 2.356 / 2.773 (85.0%)  [healthy]
Head 3: entropy = 2.384 / 2.773 (86.0%)  [healthy]
Head 4: entropy = 2.375 / 2.773 (85.7%)  [healthy]
Head 5: entropy = 2.373 / 2.773 (85.6%)  [healthy]
Head 6: entropy = 2.362 / 2.773 (85.2%)  [healthy]
Head 7: entropy = 2.328 / 2.773 (84.0%)  [healthy]


---

# Part 3: Layer Redundancy Monitoring

> 核心问题：**层是否在做重复计算？**

---

## 7. Cross-Layer Similarity (CKA)

**直觉：**  
衡量相邻层是否学习了相同特征。

**定义：**

$$
\text{CKA}(X, Y) = \frac{\|X^T Y\|_F^2}{\|X^T X\|_F \cdot \|Y^T Y\|_F}
$$

**监控意义：**

- 接近 1 → 层完全冗余  
- 接近 0 → 表达不同  

**论文出处：**

> Simon Kornblith, Mohammad Norouzi, Honglak Lee, Geoffrey Hinton.  
> *Similarity of Neural Network Representations Revisited.* ICML 2019.  
> https://proceedings.mlr.press/v97/kornblith19a.html

In [9]:
def linear_cka(X: torch.Tensor, Y: torch.Tensor, max_samples: int = 4096) -> float:
    """Linear CKA between two activation matrices X, Y ∈ (N, d).
    Subsamples rows when N is large to keep cost at O(max_samples · d^2).
    """
    X, Y = X.float(), Y.float()
    N = X.shape[0]
    if N > max_samples:
        idx = torch.randperm(N, device=X.device)[:max_samples]
        X, Y = X[idx], Y[idx]
    XtY = X.T @ Y
    XtX = X.T @ X
    YtY = Y.T @ Y
    numerator = torch.linalg.matrix_norm(XtY, ord="fro") ** 2
    denominator = torch.linalg.matrix_norm(XtX, ord="fro") * torch.linalg.matrix_norm(YtY, ord="fro")
    return (numerator / (denominator + 1e-12)).item()


# demo
net = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

layer_outputs = {}

def capture(name):
    def hook(m, inp, out):
        layer_outputs[name] = out.detach()
    return hook

net[1].register_forward_hook(capture("relu_0"))
net[3].register_forward_hook(capture("relu_1"))

x = torch.randn(128, 64)
_ = net(x)

cka = linear_cka(layer_outputs["relu_0"], layer_outputs["relu_1"])
print(f"CKA(relu_0, relu_1) = {cka:.4f}")
if cka > 0.9:
    print("ALERT: layers are nearly redundant — consider pruning.")

CKA(relu_0, relu_1) = 0.9601
ALERT: layers are nearly redundant — consider pruning.


---

# Engineering Guidelines & Diagnostics

---

## 分层监控（必须）

不要只看 global loss：

- 每层必须监控：
  - srank
  - DNR
  - WPR

---

## 硬性报警规则

- srank < 0.05 × dim → Dimensional Collapse  
- DNR > 0.8 → Layer Dead  
- GNS 突然跳变 → Noise Dominated Training  

---

## Gradient Spectrum 诊断

如果：

- 最大奇异值占比 > 90%

则：

→ 模型已发生 rank collapse

建议：

- 立即停止训练  
- 检查：
  - Learning Rate  
  - Normalization  
  - Feature Scaling  

---

# Core Insight

当网络发生深层坍缩时：

**Effective Capacity << Design Capacity**

关键监控组合：

- WPR → 容量是否被使用  
- srank → 表达是否坍缩  
- GNS → 优化是否在放大问题  

---

# Production 建议

- 使用 per-layer logging  
- 混合数据分布（normal + hard cases）  
- 在结构 collapse 时 early stop  
- 与 E2E 指标（如 ADE / trajectory drift）联动  